In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import sqlite3

In [4]:
movies = pd.read_csv("movies.csv")
genres = pd.read_csv("genres.csv")
movie_genres = pd.read_csv("movie_genres.csv")
movie_keywords = pd.read_csv("movie_keywords.csv")
cast = pd.read_csv("cast.csv")
crew = pd.read_csv("crew.csv")

movies.head()

,movie_id,title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,status,overview
0,157336,Interstellar,en,2014-11-05,169,165000000.0,7.466067e+08,73.0452,8.484,40566,Released,The adventures of a group of explorers who mak...
1,27205,Inception,en,2010-07-15,148,160000000.0,8.390306e+08,55.8497,8.400,39727,Released,"Cobb, a skilled thief who commits corporate es..."
2,24428,The Avengers,en,2012-04-25,143,220000000.0,1.518816e+09,72.2515,8.062,38993,Released,When an unexpected enemy emerges and threatens...
3,155,The Dark Knight,en,2008-07-16,152,185000000.0,1.004558e+09,60.5510,8.534,36255,Released,Batman raises the stakes in his war on crime. ...
4,19995,Avatar,en,2009-12-16,162,237000000.0,2.923706e+09,48.1996,7.610,34348,Released,"In the 22nd century, a paraplegic Marine is di..."


In [5]:
print("Movies:", movies.shape)
print("Genres:", genres.shape)
print("Movie Genres:", movie_genres.shape)
print("Movie Keywords:", movie_keywords.shape)
print("Cast:", cast.shape)
print("Crew:", crew.shape)

Movies: (2503, 12)
Genres: (19, 2)
Movie Genres: (6878, 2)
Movie Keywords: (41681, 3)
Cast: (24902, 6)
Crew: (7289, 6)


In [5]:
print("Unique Movies:", movies["movie_id"].nunique())
print("Unique Genres:", genres["genre_id"].nunique())
print("Unique Cast Members:", cast["actor_name"].nunique())
print("Unique Crew Members:", crew["person_name"].nunique())

Unique Movies: 2503
Unique Genres: 19
Unique Cast Members: 11291
Unique Crew Members: 2846


In [19]:
import numpy as np

check = movies[
    (movies["budget"].isna()) |
    (movies["revenue"].isna()) |
    (movies["budget"] == 0) |
    (movies["revenue"] == 0)
]

print("Movies with missing or zero budget/revenue:")
display(check[["title", "budget", "revenue"]])


Movies with missing or zero budget/revenue:


,title,budget,revenue
324,Zack Snyder's Justice League,70000000.0,0.0
340,Bird Box,19800000.0,0.0
403,365 Days,0.0,9458590.0
468,To All the Boys I've Loved Before,0.0,0.0
554,Prey,65000000.0,0.0
...,...,...,...
2469,Day Shift,100000000.0,0.0
2471,A Serbian Film,0.0,0.0
2480,Chungking Express,160000.0,0.0
2481,I'm Thinking of Ending Things,0.0,0.0


In [20]:
valid_movies = movies[
    (movies["budget"] > 0) &
    (movies["revenue"] > 0)
].copy()

valid_movies["profit"] = (
    valid_movies["revenue"] - valid_movies["budget"]
)

valid_movies["ROI"] = (
    valid_movies["profit"] / valid_movies["budget"]
) * 100

display(
    valid_movies[
        ["title", "budget", "revenue", "profit", "ROI"]
    ].head(10)
)

,title,budget,revenue,profit,ROI
0,Interstellar,165000000.0,7.466067e+08,5.816067e+08,352.488913
1,Inception,160000000.0,8.390306e+08,6.790306e+08,424.394144
2,The Avengers,220000000.0,1.518816e+09,1.298816e+09,590.370689
3,The Dark Knight,185000000.0,1.004558e+09,8.195584e+08,443.004564
4,Avatar,237000000.0,2.923706e+09,2.686706e+09,1133.631235
5,Deadpool,58000000.0,7.828373e+08,7.248373e+08,1249.719564
6,Fight Club,63000000.0,1.008538e+08,3.785375e+07,60.085322
7,Avengers: Infinity War,300000000.0,2.052415e+09,1.752415e+09,584.138346
8,The Shawshank Redemption,25000000.0,2.834147e+07,3.341469e+06,13.365876
9,Pulp Fiction,8000000.0,2.139288e+08,2.059288e+08,2574.109525


In [27]:
movies["release_date"] = pd.to_datetime(
    movies["release_date"],
    errors="coerce"
)

movies["year"] = movies["release_date"].dt.year

year_count = movies.groupby("year")["movie_id"].nunique()

display(year_count)


,movie_id
year,
1921,1
1922,1
1927,1
1931,2
1936,1
...,...
2022,65
2023,62
2024,43


In [29]:
runtime_data = movies[
    movies["runtime"].notna() &
    (movies["runtime"] > 0)
].copy()

Q1 = runtime_data["runtime"].quantile(0.25)
Q3 = runtime_data["runtime"].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

outliers = runtime_data[
    (runtime_data["runtime"] < lower_limit) |
    (runtime_data["runtime"] > upper_limit)
]

print("Lower Limit:", lower_limit)
print("Upper Limit:", upper_limit)
print("Total Runtime Outliers:", len(outliers))

display(outliers[["title", "runtime"]].head(20))

Lower Limit: 58.5
Upper Limit: 166.5
Total Runtime Outliers: 60


,title,runtime
0,Interstellar,169
16,Avengers: Endgame,181
18,The Lord of the Rings: The Fellowship of the Ring,179
19,Titanic,194
20,The Lord of the Rings: The Return of the King,201
21,The Wolf of Wall Street,180
28,The Lord of the Rings: The Two Towers,179
38,The Godfather,175
76,The Hobbit: An Unexpected Journey,169
79,The Green Mile,189


In [12]:
genre_analysis = movie_genres.merge(
    genres,
    on="genre_id"
)

genre_analysis = genre_analysis.merge(
    movies[["movie_id", "title", "vote_average"]],
    on="movie_id"
)

genre_summary = genre_analysis.groupby(
    "genre_name"
).agg(
    total_movies=("movie_id", "nunique"),
    average_rating=("vote_average", "mean")
).reset_index()

genre_summary = genre_summary.sort_values(
    "total_movies",
    ascending=False
)

display(genre_summary)

,genre_name,total_movies,average_rating
6,Drama,937,7.252454
0,Action,808,6.729059
3,Comedy,762,6.809316
1,Adventure,674,6.834755
16,Thriller,673,6.861853
14,Science Fiction,463,6.814102
8,Fantasy,396,6.825242
4,Crime,387,7.006690
13,Romance,349,7.033762
7,Family,344,6.935552


In [13]:
top_rated = movies[
    movies["vote_average"].notna() &
    (movies["vote_count"] >= 100)
].sort_values(
    by="vote_average",
    ascending=False
).head(10)

display(
    top_rated[
        ["title", "vote_average", "vote_count"]
    ]
)

,title,vote_average,vote_count
8,The Shawshank Redemption,8.727,30915
1465,Michael,8.696,3846
38,The Godfather,8.686,23255
713,Project Hail Mary,8.659,6750
108,Schindler's List,8.570,17680
187,The Godfather Part II,8.570,14116
373,12 Angry Men,8.566,10192
3,The Dark Knight,8.534,36255
91,Spirited Away,8.533,18619
1171,Dilwale Dulhania Le Jayenge,8.507,4607


In [37]:
actor_analysis = cast.groupby(
    "actor_name"
)["movie_id"].nunique().reset_index()

actor_analysis.columns = [
    "actor_name",
    "movie_count"
]

top_actors = actor_analysis.sort_values(
    "movie_count",
    ascending=False
).head(10)

display(top_actors)

,actor_name,movie_count
9526,Samuel L. Jackson,39
9102,Robert De Niro,38
1255,Brad Pitt,38
5317,Johnny Depp,37
10599,Tom Hanks,33
11031,Willem Dafoe,32
7090,Mark Wahlberg,32
9630,Scarlett Johansson,32
10586,Tom Cruise,31
7814,Morgan Freeman,31


In [15]:
top_revenue = movies[
    movies["revenue"] > 0
].sort_values(
    by="revenue",
    ascending=False
).head(10)

display(
    top_revenue[
        ["title", "budget", "revenue", "vote_average"]
    ]
)

,title,budget,revenue,vote_average
4,Avatar,237000000.0,2.923706e+09,7.610
16,Avengers: Endgame,356000000.0,2.799439e+09,8.238
178,Avatar: The Way of Water,460000000.0,2.334485e+09,7.600
19,Titanic,200000000.0,2.264162e+09,7.901
65,Star Wars: The Force Awakens,245000000.0,2.068224e+09,7.247
7,Avengers: Infinity War,300000000.0,2.052415e+09,8.240
46,Spider-Man: No Way Home,200000000.0,1.921207e+09,7.934
1776,Zootopia 2,150000000.0,1.868209e+09,7.680
695,Inside Out 2,200000000.0,1.698864e+09,7.543
53,Jurassic World,150000000.0,1.671537e+09,6.700
